In [14]:
# ETAP 1: Wczytanie i Wstępna Selekcja Danych
# Skrypt weryfikuje obecność pliku źródłowego z danymi Oracle's Elixir, ładuje go
# do pamięci (omijając ewentualne błędy strukturalne w pliku) i modyfikuje ustawienia
# wyświetlania pandas. Następnie wycina z pełnego zbioru danych tylko te kluczowe kolumny,
# które posłużą nam do zbudowania Asystenta Draftu.

import pandas as pd

# Sama nazwa pliku - skoro leży w tym samym folderze co notatnik z kodem
nazwa_pliku = '2026_LoL_esports_match_data_from_OraclesElixir.csv' 

print("Wczytywanie lokalnego pliku...")
df = pd.read_csv(nazwa_pliku, low_memory=False, on_bad_lines='warn')
print("Dane wczytane pomyślnie!")

pd.set_option('display.max_columns', None)

podstawowe = ['league', 'side', 'position', 'firstPick', 'champion', 'result',]
bany = [f'ban{i}' for i in range(1, 6)]

df_cleaned = df[podstawowe + bany]
display(df_cleaned.head())


Wczytywanie lokalnego pliku...
Dane wczytane pomyślnie!


,league,side,position,firstPick,champion,result,ban1,ban2,ban3,ban4,ban5
0,LIT,Blue,top,1.0,Gnar,0,Zyra,Lillia,Wukong,Veigar,Viktor
1,LIT,Blue,jng,1.0,Xin Zhao,0,Zyra,Lillia,Wukong,Veigar,Viktor
2,LIT,Blue,mid,1.0,Ahri,0,Zyra,Lillia,Wukong,Veigar,Viktor
3,LIT,Blue,bot,1.0,Corki,0,Zyra,Lillia,Wukong,Veigar,Viktor
4,LIT,Blue,sup,1.0,Leona,0,Zyra,Lillia,Wukong,Veigar,Viktor


In [15]:
# ETAP 2: Filtrowanie Wierszy (Obserwacji)
# Zawężamy zbiór danych wyłącznie do głównych profesjonalnych lig e-sportowych.
# Dodatkowo usuwamy wiersze ze zbiorczymi statystykami drużyn (position == 'team'),
# pozostawiając w bazie jedynie indywidualne występy zawodników na poszczególnych liniach.

glowne_ligi = ['LEC', 'LCK', 'LPL', 'LCS']
df_cleaned = df_cleaned[df_cleaned['league'].isin(glowne_ligi)]

df_cleaned = df_cleaned[df_cleaned['position'] != 'team']

In [16]:
# ETAP 3: Kodowanie Zmiennych Kategorycznych (One-Hot Encoding)
# Zmieniamy nazwy bohaterów ze stringów na format liczbowy (zera i jedynki).
# Tworzymy osobne kolumny dla każdej postaci (dummy variables), co zapobiegnie
# nadaniu postaciom sztucznej hierarchii matematycznej i pozwoli sieci 
# neuronowej poprawnie zinterpretować, kto został wybrany lub zablokowany

wszyscy_bohaterowie = [
    'Aatrox', 'Ahri', 'Akali', 'Akshan', 'Alistar', 'Ambessa', 'Amumu', 'Anivia', 'Annie', 'Aphelios', 'Ashe', 
    'Aurelion Sol', 'Aurora', 'Azir', 'Bard', 'Bel\'Veth', 'Blitzcrank', 'Brand', 'Braum', 'Briar', 'Caitlyn', 
    'Camille', 'Cassiopeia', 'Cho\'Gath', 'Corki', 'Darius', 'Diana', 'Dr. Mundo', 'Draven', 'Ekko', 'Elise', 
    'Evelynn', 'Ezreal', 'Fiddlesticks', 'Fiora', 'Fizz', 'Galio', 'Gangplank', 'Garen', 'Gnar', 'Gragas', 
    'Graves', 'Gwen', 'Hecarim', 'Heimerdinger', 'Hwei', 'Illaoi', 'Irelia', 'Ivern', 'Janna', 'Jarvan IV', 
    'Jax', 'Jayce', 'Jhin', 'Jinx', 'K\'Sante', 'Kai\'Sa', 'Kalista', 'Karma', 'Karthus', 'Kassadin', 'Katarina', 
    'Kayle', 'Kayn', 'Kennen', 'Kha\'Zix', 'Kindred', 'Kled', 'Kog\'Maw', 'LeBlanc', 'Lee Sin', 'Leona', 
    'Lillia', 'Lissandra', 'Lucian', 'Lulu', 'Lux', 'Malphite', 'Malzahar', 'Maokai', 'Master Yi', 'Mel', 
    'Milio', 'Miss Fortune', 'Mordekaiser', 'Morgana', 'Naafiri', 'Nami', 'Nasus', 'Nautilus', 'Neeko', 
    'Nidalee', 'Nilah', 'Nocturne', 'Nunu & Willump', 'Olaf', 'Orianna', 'Ornn', 'Pantheon', 'Poppy', 'Pyke', 
    'Qiyana', 'Quinn', 'Rakan', 'Rammus', 'Rek\'Sai', 'Rell', 'Renata Glasc', 'Renekton', 'Rengar', 'Riven', 
    'Rumble', 'Ryze', 'Samira', 'Sejuani', 'Senna', 'Seraphine', 'Sett', 'Shaco', 'Shen', 'Shyvana', 'Singed', 
    'Sion', 'Sivir', 'Skarner', 'Smolder', 'Sona', 'Soraka', 'Swain', 'Sylas', 'Syndra', 'Tahm Kench', 'Taliyah', 
    'Talon', 'Taric', 'Teemo', 'Thresh', 'Tristana', 'Trundle', 'Tryndamere', 'Twisted Fate', 'Twitch', 'Udyr', 
    'Urgot', 'Varus', 'Vayne', 'Veigar', 'Vel\'Koz', 'Vex', 'Vi', 'Viego', 'Viktor', 'Vladimir', 'Volibear', 
    'Warwick', 'Wukong', 'Xayah', 'Xerath', 'Xin Zhao', 'Yasuo', 'Yone', 'Yorick', 'Yuumi', 'Yunara', 'Zaahen', 
    'Zac', 'Zed', 'Zeri', 'Ziggs', 'Zilean', 'Zoe', 'Zyra'
]

bany = ['ban1','ban2','ban3','ban4','ban5']
kolumny_do_zmiany = ['champion'] + bany

# Nadpisujemy kolumny, wymuszając nasze sztywne kategorie
for kolumna in kolumny_do_zmiany:
    df_cleaned[kolumna] = pd.Categorical(df_cleaned[kolumna], categories=wszyscy_bohaterowie)

# Tworzymy ostateczną macierz z zerami i jedynkami
df_encoded = pd.get_dummies(df_cleaned, columns=kolumny_do_zmiany, dtype=int)

print(f"Nowy rozmiar tabeli: {df_encoded.shape}")
display(df_encoded.head())



Nowy rozmiar tabeli: (14180, 1037)


C:\Users\HZCWXX\AppData\Local\Temp\ipykernel_24412\3310847888.py:32: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df_cleaned[kolumna] = pd.Categorical(df_cleaned[kolumna], categories=wszyscy_bohaterowie)


,league,side,position,firstPick,result,champion_Aatrox,champion_Ahri,champion_Akali,champion_Akshan,champion_Alistar,champion_Ambessa,champion_Amumu,champion_Anivia,champion_Annie,champion_Aphelios,champion_Ashe,champion_Aurelion Sol,champion_Aurora,champion_Azir,champion_Bard,champion_Bel'Veth,champion_Blitzcrank,champion_Brand,champion_Braum,champion_Briar,champion_Caitlyn,champion_Camille,champion_Cassiopeia,champion_Cho'Gath,champion_Corki,champion_Darius,champion_Diana,champion_Dr. Mundo,champion_Draven,champion_Ekko,champion_Elise,champion_Evelynn,champion_Ezreal,champion_Fiddlesticks,champion_Fiora,champion_Fizz,champion_Galio,champion_Gangplank,champion_Garen,champion_Gnar,champion_Gragas,champion_Graves,champion_Gwen,champion_Hecarim,champion_Heimerdinger,champion_Hwei,champion_Illaoi,champion_Irelia,champion_Ivern,champion_Janna,champion_Jarvan IV,champion_Jax,champion_Jayce,champion_Jhin,champion_Jinx,champion_K'Sante,champion_Kai'Sa,champion_Kalista,champion_Karma,champion_Karthus,champion_Kassadin,champion_Katarina,champion_Kayle,champion_Kayn,champion_Kennen,champion_Kha'Zix,champion_Kindred,champion_Kled,champion_Kog'Maw,champion_LeBlanc,champion_Lee Sin,champion_Leona,champion_Lillia,champion_Lissandra,champion_Lucian,champion_Lulu,champion_Lux,champion_Malphite,champion_Malzahar,champion_Maokai,champion_Master Yi,champion_Mel,champion_Milio,champion_Miss Fortune,champion_Mordekaiser,champion_Morgana,champion_Naafiri,champion_Nami,champion_Nasus,champion_Nautilus,champion_Neeko,champion_Nidalee,champion_Nilah,champion_Nocturne,champion_Nunu & Willump,champion_Olaf,champion_Orianna,champion_Ornn,champion_Pantheon,champion_Poppy,champion_Pyke,champion_Qiyana,champion_Quinn,champion_Rakan,champion_Rammus,champion_Rek'Sai,champion_Rell,champion_Renata Glasc,champion_Renekton,champion_Rengar,champion_Riven,champion_Rumble,champion_Ryze,champion_Samira,champion_Sejuani,champion_Senna,champion_Seraphine,champion_Sett,champion_Shaco,champion_Shen,champion_Shyvana,champion_Singed,champion_Sion,champion_Sivir,champion_Skarner,champion_Smolder,champion_Sona,champion_Soraka,champion_Swain,champion_Sylas,champion_Syndra,champion_Tahm Kench,champion_Taliyah,champion_Talon,champion_Taric,champion_Teemo,champion_Thresh,champion_Tristana,champion_Trundle,champion_Tryndamere,champion_Twisted Fate,champion_Twitch,champion_Udyr,champion_Urgot,champion_Varus,champion_Vayne,champion_Veigar,champion_Vel'Koz,champion_Vex,champion_Vi,champion_Viego,champion_Viktor,champion_Vladimir,champion_Volibear,champion_Warwick,champion_Wukong,champion_Xayah,champion_Xerath,champion_Xin Zhao,champion_Yasuo,champion_Yone,champion_Yorick,champion_Yuumi,champion_Yunara,champion_Zaahen,champion_Zac,champion_Zed,champion_Zeri,champion_Ziggs,champion_Zilean,champion_Zoe,champion_Zyra,ban1_Aatrox,ban1_Ahri,ban1_Akali,ban1_Akshan,ban1_Alistar,ban1_Ambessa,ban1_Amumu,ban1_Anivia,ban1_Annie,ban1_Aphelios,ban1_Ashe,ban1_Aurelion Sol,ban1_Aurora,ban1_Azir,ban1_Bard,ban1_Bel'Veth,ban1_Blitzcrank,ban1_Brand,ban1_Braum,ban1_Briar,ban1_Caitlyn,ban1_Camille,ban1_Cassiopeia,ban1_Cho'Gath,ban1_Corki,ban1_Darius,ban1_Diana,ban1_Dr. Mundo,ban1_Draven,ban1_Ekko,ban1_Elise,ban1_Evelynn,ban1_Ezreal,ban1_Fiddlesticks,ban1_Fiora,ban1_Fizz,ban1_Galio,ban1_Gangplank,ban1_Garen,ban1_Gnar,ban1_Gragas,ban1_Graves,ban1_Gwen,ban1_Hecarim,ban1_Heimerdinger,ban1_Hwei,ban1_Illaoi,ban1_Irelia,ban1_Ivern,ban1_Janna,ban1_Jarvan IV,ban1_Jax,ban1_Jayce,ban1_Jhin,ban1_Jinx,ban1_K'Sante,ban1_Kai'Sa,ban1_Kalista,ban1_Karma,ban1_Karthus,ban1_Kassadin,ban1_Katarina,ban1_Kayle,ban1_Kayn,ban1_Kennen,ban1_Kha'Zix,ban1_Kindred,ban1_Kled,ban1_Kog'Maw,ban1_LeBlanc,ban1_Lee Sin,ban1_Leona,ban1_Lillia,ban1_Lissandra,ban1_Lucian,ban1_Lulu,ban1_Lux,ban1_Malphite,ban1_Malzahar,ban1_Maokai,ban1_Master Yi,ban1_Mel,ban1_Milio,ban1_Miss Fortune,ban1_Mordekaiser,ban1_Morgana,ban1_Naafiri,ban1_Nami,ban1_Nasus,ban1_Nautilus,ban1_Neeko,ban1_Nidalee,ban1_Nilah,ban1_Nocturne,ban1_Nunu & Willump,ban1_Olaf,ban1_Ori

In [17]:

# ETAP 4: Separacja Cech i Podział na Zbiory (Train/Test Split)
# Konwertujemy pozostałe kolumny kategoryczne (liga, strona, rola) 
# na format numeryczny, oddzielamy macierz cech (X) od wyniku meczu (y)
# i dzielimy dane na zbiór treningowy (80%) oraz testowy (20%).


from sklearn.model_selection import train_test_split

# Kodujemy pozostałe kolumny tekstowe na zera i jedynki
kolumny_pozostale = ['league', 'side', 'position', 'firstPick']
df_encoded = pd.get_dummies(df_encoded, columns=kolumny_pozostale, dtype=int)

# Rozdzielenie na X (cechy wejściowe) i y (wynik meczu)
y = df_encoded['result']
X = df_encoded.drop(columns=['result'])

# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Liczba próbek treningowych: {X_train.shape[0]}, cech wejściowych: {X_train.shape[1]}")
print(f"Liczba próbek testowych: {X_test.shape[0]}")

Liczba próbek treningowych: 11344, cech wejściowych: 1045
Liczba próbek testowych: 2836


In [18]:
# ETAP 5: Definicja Architektury Sieci w PyTorch (MLP)

import torch
import torch.nn as nn

class DraftPredictor(nn.Module):
    def __init__(self, input_dim):
        super(DraftPredictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.net(x)

input_dim = X_train.shape[1]
model = DraftPredictor(input_dim)
print(model)


DraftPredictor(
  (net): Sequential(
    (0): Linear(in_features=1045, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)


In [19]:

# ETAP 6: Konwersja na Tensory i Trening w PyTorch

import torch.optim as optim

# 1. Konwersja danych pandas -> PyTorch Tensor (float32)
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# 2. Definicja funkcji straty i optymalizatora
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Pętla treningowa (30 epok)
epochs = 30
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()              
    
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor) 
    
    loss.backward()                     
    optimizer.step()                    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            test_preds = model(X_test_tensor)
            test_loss = criterion(test_preds, y_test_tensor)
            acc = ((test_preds >= 0.5) == y_test_tensor).float().mean().item() * 100
        print(f"Epoka [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | Test Loss: {test_loss.item():.4f} | Test Acc: {acc:.2f}%")

Epoka [1/30] | Loss: 0.6936 | Test Loss: 0.6929 | Test Acc: 50.49%
Epoka [5/30] | Loss: 0.6923 | Test Loss: 0.6921 | Test Acc: 52.33%
Epoka [10/30] | Loss: 0.6896 | Test Loss: 0.6894 | Test Acc: 62.31%
Epoka [15/30] | Loss: 0.6831 | Test Loss: 0.6829 | Test Acc: 63.19%
Epoka [20/30] | Loss: 0.6712 | Test Loss: 0.6722 | Test Acc: 64.14%
Epoka [25/30] | Loss: 0.6531 | Test Loss: 0.6568 | Test Acc: 66.18%
Epoka [30/30] | Loss: 0.6291 | Test Loss: 0.6389 | Test Acc: 66.47%


In [20]:

# ETAP 7: Ewaluacja Modelu (Confusion Matrix & Classification Report)

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
with torch.no_grad():
    raw_preds = model(X_test_tensor)
    y_pred_binary = (raw_preds >= 0.5).int().numpy()
    y_true = y_test_tensor.int().numpy()

print("Raport Klasyfikacji:")
print(classification_report(y_true, y_pred_binary, target_names=['Porażka (0)', 'Wygrana (1)']))

cm = confusion_matrix(y_true, y_pred_binary)
print("Macierz pomyłek (Confusion Matrix):")
print(f"Prawdziwe Porażki (TN): {cm[0,0]} | Fałszywe Wygrane (FP): {cm[0,1]}")
print(f"Fałszywe Porażki (FN): {cm[1,0]} | Prawdziwe Wygrane (TP): {cm[1,1]}")

Raport Klasyfikacji:
              precision    recall  f1-score   support

 Porażka (0)       0.68      0.62      0.65      1432
 Wygrana (1)       0.65      0.71      0.68      1404

    accuracy                           0.66      2836
   macro avg       0.67      0.67      0.66      2836
weighted avg       0.67      0.66      0.66      2836

Macierz pomyłek (Confusion Matrix):
Prawdziwe Porażki (TN): 891 | Fałszywe Wygrane (FP): 541
Fałszywe Porażki (FN): 410 | Prawdziwe Wygrane (TP): 994


In [21]:
# ETAP 8: Interaktywny Symulator Draftu (Prediction Engine)

import pandas as pd
import torch

def predict_draft_winrate(blue_picks, red_picks, blue_bans=None, red_bans=None, league='LEC'):
    """
    Oblicza prawdopodobieństwo wygranej dla Blue Side na podstawie kompozycji draftu.
    """
    model.eval()
    
    if blue_bans is None: blue_bans = []
    if red_bans is None: red_bans = []
    
    # 1. Tworzymy pustą ramkę danych z dokładnie takimi samymi kolumnami jak w X
    input_row = pd.DataFrame(0, index=[0], columns=X.columns)
    
    # 2. Ustawiamy flagi strony (analiza z perspektywy Blue Side)
    if 'side_Blue' in input_row.columns:
        input_row['side_Blue'] = 1
    if 'firstPick_1.0' in input_row.columns:
        input_row['firstPick_1.0'] = 1
    if f'league_{league}' in input_row.columns:
        input_row[f'league_{league}'] = 1
        
    # 3. Ustawiamy picki dla Blue Side
    for champ in blue_picks:
        col = f'champion_{champ}'
        if col in input_row.columns:
            input_row[col] = 1
            
    # 4. Ustawiamy bany (jeśli zostały podane)
    for i, ban in enumerate(blue_bans[:5], start=1):
        col = f'ban{i}_{ban}'
        if col in input_row.columns:
            input_row[col] = 1

    # 5. Konwersja do Tensora i predykcja przez model
    input_tensor = torch.tensor(input_row.values, dtype=torch.float32)
    
    with torch.no_grad():
        win_prob = model(input_tensor).item()
        
    blue_winrate = win_prob * 100
    red_winrate = (1 - win_prob) * 100
    
    print("=" * 45)
    print("           WYNIK SYMULACJI DRAFTU           ")
    print("=" * 45)
    print(f"🔵 BLUE SIDE: {blue_winrate:.2f}% szans na wygraną")
    print(f"🔴 RED SIDE:  {red_winrate:.2f}% szans na wygraną")
    print("-" * 45)
    if blue_winrate > 50:
        print(f"Faworyt: 🔵 BLUE SIDE (+{blue_winrate - 50:.2f}%)")
    else:
        print(f"Faworyt: 🔴 RED SIDE (+{red_winrate - 50:.2f}%)")
    print("=" * 45)
    
    return win_prob

In [22]:

# ETAP 9: Silnik Rekomendacji Picków (Draft Recommender - Greedy Search)

import pandas as pd
import torch

def find_best_pick(current_blue_picks, current_red_picks, blue_bans=None, red_bans=None, league='LEC', top_n=5):
    """
    Testuje wszystkich dostępnych championów w grze i zwraca TOP N rekomendacji,
    które maksymalizują szanse na wygraną Blue Side.
    """
    model.eval()
    
    if blue_bans is None: blue_bans = []
    if red_bans is None: red_bans = []
    
    # 1. Wyciągamy z kolumn modelu listę wszystkich unikalnych championów
    all_champions = [
        col.replace('champion_', '') 
        for col in X.columns 
        if col.startswith('champion_')
    ]
    
    # 2. Tworzymy zbiór postaci już niedostępnych (wybranych lub zbanowanych)
    unavailable_champions = set(current_blue_picks + current_red_picks + blue_bans + red_bans)
    
    # 3. Wyznaczamy wolną pulę bohaterów
    available_champions = [c for c in all_champions if c not in unavailable_champions]
    
    recommendations = []
    
    # 4. Sprawdzamy każdego wolnego bohatera
    for candidate in available_champions:
        # Tworzymy symulowany skład z dodanym kandydatem
        simulated_blue = current_blue_picks + [candidate]
        
        # Budujemy wiersz wejściowy
        input_row = pd.DataFrame(0, index=[0], columns=X.columns)
        
        if 'side_Blue' in input_row.columns:
            input_row['side_Blue'] = 1
        if 'firstPick_1.0' in input_row.columns:
            input_row['firstPick_1.0'] = 1
        if f'league_{league}' in input_row.columns:
            input_row[f'league_{league}'] = 1
            
        for champ in simulated_blue:
            col = f'champion_{champ}'
            if col in input_row.columns:
                input_row[col] = 1
                
        for i, ban in enumerate(blue_bans[:5], start=1):
            col = f'ban{i}_{ban}'
            if col in input_row.columns:
                input_row[col] = 1
                
        # Predykcja
        input_tensor = torch.tensor(input_row.values, dtype=torch.float32)
        with torch.no_grad():
            win_prob = model(input_tensor).item()
            
        recommendations.append({
            'champion': candidate,
            'predicted_blue_winrate': round(win_prob * 100, 2)
        })
        
    # 5. Tworzymy DataFrame i sortujemy od najwyższego winrate'u
    df_recs = pd.DataFrame(recommendations)
    df_sorted = df_recs.sort_values(by='predicted_blue_winrate', ascending=False).reset_index(drop=True)
    
    # Formatowanie i prezentacja wyników
    print("=" * 50)
    print(f" TOP {top_n} REKOMENDACJI PICKA DLA 🔵 BLUE SIDE ({league})")
    print("=" * 50)
    print(f"Aktualny skład Blue: {current_blue_picks}")
    print(f"Bany: {blue_bans + red_bans}")
    print("-" * 50)
    for idx, row in df_sorted.head(top_n).iterrows():
        print(f" #{idx+1} | {row['champion']:<16} -> Szansa wygranej: {row['predicted_blue_winrate']:.2f}%")
    print("=" * 50)
    
    return df_sorted.head(top_n)

In [23]:
# ETAP 10: Zapis Modelu i Kolumn dla Aplikacji Webowej (Streamlit)
import json
import torch

torch.save(model.state_dict(), 'lol_draft_model.pth')
with open('model_columns.json', 'w') as f:
    json.dump(list(X.columns), f)

print("Gotowe! Pliki zapisane na dysku.")

Gotowe! Pliki zapisane na dysku.


In [25]:
!c:\Users\HZCWXX\AppData\Local\Programs\Python\Python314\Scripts\streamlit.exe run app.py

^C
